# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anujrkt06-tech/Flyrank-ML-project-/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.


## 1. Method choice and why

I use a Random Forest classifier because the target is whether a page is declining, and the relationship between traffic, position, CTR, content age, and engagement may not be purely linear. Random Forest can capture these interactions while still giving feature-importance information for interpretation.

The model is used for decision-support: it ranks pages by estimated likelihood of decline rather than proving that a feature causes decline.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# ML-08 setup + method check

import os
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score

# Load the anonymized starter data
DATA_PATH = "data/raw/content_refresh_anonymized.csv"

if not os.path.exists(DATA_PATH):
    !git clone -q https://github.com/anujrkt06-tech/Flyrank-ML-project-.git
    DATA_PATH = "Flyrank-ML-project-/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)

print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Target available:", "is_declining_label" in df.columns)

# Create log features used by the modeling toolkit if they are not already present
for source, target in [
    ("impressions_90d", "log_impressions_90d"),
    ("clicks_90d", "log_clicks_90d"),
    ("sessions_90d", "log_sessions_90d"),
    ("ai_sessions_90d", "log_ai_sessions_90d"),
]:
    if source in df.columns and target not in df.columns:
        df[target] = np.log1p(pd.to_numeric(df[source], errors="coerce"))

print("\nRandom Forest will be used as the main model.")
print("Target:", "is_declining_label")

Rows: 30000
Columns: 44
Target available: False

Random Forest will be used as the main model.
Target: is_declining_label


## 2. Split design

I use a client-aware holdout split. Rows from the same client should not be placed in both training and test data because that could make the evaluation look easier than it really is.

About 80% of clients are used for training and the remaining clients are held out for testing. This gives a more honest estimate of how the model may perform on unseen clients.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Client-aware train/test split

# ML-08 — 2. Split design
# Create the decline label from the available trend_direction field.

RANDOM_STATE = 42

# The raw dataset does not contain is_declining_label.
# We create the bootstrap decline label from trend_direction.
if "is_declining_label" not in df.columns:
    if "trend_direction" not in df.columns:
        raise KeyError("Neither is_declining_label nor trend_direction exists in the dataset.")

    df["is_declining_label"] = (
        df["trend_direction"]
        .astype(str)
        .str.lower()
        .eq("down")
        .astype(int)
    )

target = df["is_declining_label"].astype(int)

# Client-aware split
if "client_id" not in df.columns:
    raise KeyError("client_id column is required for the client-aware split.")

client_series = df["client_id"].fillna("unknown").astype(str)

unique_clients = client_series.drop_duplicates().to_numpy()

rng = np.random.default_rng(RANDOM_STATE)
shuffled_clients = rng.permutation(unique_clients)

test_client_count = max(
    1,
    int(round(len(shuffled_clients) * 0.20))
)

test_clients = set(
    shuffled_clients[:test_client_count]
)

test_mask = client_series.isin(test_clients)

train_idx = np.where(~test_mask)[0]
test_idx = np.where(test_mask)[0]

# Safety checks
if len(train_idx) == 0 or len(test_idx) == 0:
    raise ValueError("Train/test split failed.")

if target.iloc[train_idx].nunique() < 2:
    raise ValueError("Training data has only one target class.")

if target.iloc[test_idx].nunique() < 2:
    raise ValueError("Test data has only one target class.")

print("Split strategy: client_holdout")
print("Training rows:", len(train_idx))
print("Test rows:", len(test_idx))
print("Training clients:", client_series.iloc[train_idx].nunique())
print("Test clients:", client_series.iloc[test_idx].nunique())

print("\nDeclining rate:")
print("Train:", round(target.iloc[train_idx].mean(), 4))
print("Test :", round(target.iloc[test_idx].mean(), 4))

# Confirm that no client leaks between train and test
assert set(
    client_series.iloc[train_idx]
).isdisjoint(
    set(client_series.iloc[test_idx])
)

print("\nCheck passed: no client appears in both train and test.")
print("\nTarget created from trend_direction == 'down'.")
print("Target distribution:")
print(target.value_counts())

Split strategy: client_holdout
Training rows: 27675
Test rows: 2325
Training clients: 26
Test clients: 6

Declining rate:
Train: 0.5548
Test : 0.391

Check passed: no client appears in both train and test.

Target created from trend_direction == 'down'.
Target distribution:
is_declining_label
1    16262
0    13738
Name: count, dtype: int64



## 3. Train + compare vs my baseline

I compare the learned models with the Week-4 baseline using the same decision-support target and Precision@50. Precision@50 measures the share of declining pages among the 50 highest-ranked pages.

The comparison uses the same client-aware test split for both the baseline and the learned models. This keeps the comparison fair.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Build the feature matrix and compare models with the Week-4 baseline

from sklearn.metrics import roc_auc_score, average_precision_score

# Numeric and categorical features from the FlyRank modeling toolkit.
numeric_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "log_impressions_90d",
    "log_clicks_90d",
    "log_sessions_90d",
    "log_ai_sessions_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
]

categorical_features = [
    "competition_level",
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "impression_tier",
    "position_tier",
]

numeric_features = [c for c in numeric_features if c in df.columns]
categorical_features = [c for c in categorical_features if c in df.columns]

X_num = df[numeric_features].apply(pd.to_numeric, errors="coerce")
X_num = X_num.replace([np.inf, -np.inf], np.nan).fillna(0)

X_cat = df[categorical_features].fillna("unknown").astype(str)
X_cat = pd.get_dummies(
    X_cat,
    prefix=categorical_features,
    dtype=float
)

X = pd.concat(
    [X_num.reset_index(drop=True), X_cat.reset_index(drop=True)],
    axis=1
)

y = target.reset_index(drop=True)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]


# -----------------------------
# Week-4 style baseline score
# -----------------------------

def percentile_rank(series):
    values = pd.to_numeric(series, errors="coerce").fillna(0)
    return values.rank(method="average", pct=True).fillna(0)

def normalize(series):
    values = pd.to_numeric(series, errors="coerce").fillna(0)
    minimum = values.min()
    maximum = values.max()

    if maximum == minimum:
        return pd.Series(np.zeros(len(values)), index=values.index)

    return (values - minimum) / (maximum - minimum)


baseline = pd.DataFrame(index=df.index)

baseline["visibility_score"] = percentile_rank(
    np.log1p(df["impressions_90d"])
)

baseline["freshness_risk_score"] = percentile_rank(
    df["days_since_last_update"]
)

baseline["position_opportunity_score"] = (
    (1 - normalize(df["avg_position"].clip(lower=1, upper=50)))
    * baseline["visibility_score"]
    * (df["avg_position"] > 0).astype(int)
)

baseline["depth_gap_score"] = (
    (1 - percentile_rank(df["word_count"]))
    * baseline["visibility_score"]
)

baseline["baseline_refresh_score"] = (
    0.40 * baseline["visibility_score"]
    + 0.30 * baseline["freshness_risk_score"]
    + 0.25 * baseline["position_opportunity_score"]
    + 0.05 * baseline["depth_gap_score"]
).clip(0, 1)


# -----------------------------
# Precision@50 helper
# -----------------------------

def precision_at_50(y_true, scores):
    temp = pd.DataFrame({
        "y": np.asarray(y_true),
        "score": np.asarray(scores)
    })

    top50 = temp.sort_values(
        "score",
        ascending=False
    ).head(50)

    return float(top50["y"].mean())


# Baseline score on the same test split
baseline_test_scores = baseline.iloc[test_idx]["baseline_refresh_score"].to_numpy()

baseline_p50 = precision_at_50(
    y_test,
    baseline_test_scores
)


# -----------------------------
# Models
# -----------------------------

models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            class_weight="balanced",
            max_iter=1000,
            random_state=RANDOM_STATE
        ))
    ]),

    "Decision Tree": DecisionTreeClassifier(
        class_weight="balanced",
        max_depth=5,
        min_samples_leaf=50,
        random_state=RANDOM_STATE
    ),

    "Random Forest": RandomForestClassifier(
        class_weight="balanced_subsample",
        max_depth=10,
        min_samples_leaf=25,
        n_estimators=200,
        n_jobs=-1,
        random_state=RANDOM_STATE
    )
}


results = []

# Baseline result
results.append({
    "method": "Week-4 baseline",
    "precision_at_50": baseline_p50
})


# Train and evaluate each model
trained_models = {}

for name, model in models.items():

    model.fit(X_train, y_train)

    probabilities = model.predict_proba(X_test)[:, 1]

    p50 = precision_at_50(
        y_test,
        probabilities
    )

    predictions = (probabilities >= 0.5).astype(int)

    results.append({
        "method": name,
        "precision_at_50": p50
    })

    trained_models[name] = model


comparison = pd.DataFrame(results)

comparison["precision_at_50"] = comparison[
    "precision_at_50"
].round(3)

print("Model comparison:")
display(comparison.sort_values(
    "precision_at_50",
    ascending=False
).reset_index(drop=True))


# Select best model by Precision@50
best_model_name = comparison.sort_values(
    "precision_at_50",
    ascending=False
).iloc[0]["method"]

print("\nBest method by measured Precision@50:", best_model_name)
print("Test split:", len(test_idx), "rows")
print("Metric: Precision@50")

Model comparison:


,method,precision_at_50
0,Random Forest,0.74
1,Decision Tree,0.58
2,Logistic Regression,0.40
3,Week-4 baseline,0.24



Best method by measured Precision@50: Random Forest
Test split: 2325 rows
Metric: Precision@50



## 4. Errors and interpretation

The main errors are false positives and false negatives around the ranking boundary. A high model score does not guarantee that a page is actually declining, and a lower score does not guarantee that a declining page will be missed.

I inspect the strongest features to understand what the model leans on. These are directional signals for decision-support, not causal explanations.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Error analysis and feature interpretation

# Use the Random Forest for interpretation because it is the selected
# modeling method for this lane.
rf = trained_models["Random Forest"]

test_probabilities = rf.predict_proba(X_test)[:, 1]
test_predictions = (test_probabilities >= 0.5).astype(int)

error_frame = df.iloc[test_idx][
    ["content_id", "client_id"]
].copy()

error_frame["actual"] = y_test.to_numpy()
error_frame["predicted"] = test_predictions
error_frame["probability"] = test_probabilities

error_frame["error_type"] = np.select(
    [
        (error_frame["actual"] == 1) & (error_frame["predicted"] == 0),
        (error_frame["actual"] == 0) & (error_frame["predicted"] == 1)
    ],
    [
        "false_negative",
        "false_positive"
    ],
    default="correct"
)

print("Error counts:")
print(error_frame["error_type"].value_counts())

print("\nModel metrics at probability threshold 0.5:")
print("Precision:", round(
    precision_score(y_test, test_predictions, zero_division=0), 3
))
print("Recall :", round(
    recall_score(y_test, test_predictions, zero_division=0), 3
))
print("F1 :", round(
    f1_score(y_test, test_predictions, zero_division=0), 3
))

# Top feature importance
importance = pd.DataFrame({
    "feature": X.columns,
    "importance": rf.feature_importances_
}).sort_values(
    "importance",
    ascending=False
).head(10)

print("\nTop 10 model features:")
display(importance.reset_index(drop=True))


# Show a small sample of false positives and false negatives.
# IDs are hashed/anonymized and are not used in the interpretation text.
print("\nFalse-positive examples:")
display(
    error_frame[
        error_frame["error_type"] == "false_positive"
    ][["actual", "predicted", "probability"]]
    .head(5)
)

print("\nFalse-negative examples:")
display(
    error_frame[
        error_frame["error_type"] == "false_negative"
    ][["actual", "predicted", "probability"]]
    .head(5)
)

print("\nInterpretation:")
print(
    "The model should be treated as decision-support. "
    "Feature importance shows which available signals the Random Forest "
    "used most strongly, but it does not establish causation."
)

Error counts:
error_type
correct           1563
false_positive     529
false_negative     233
Name: count, dtype: int64

Model metrics at probability threshold 0.5:
Precision: 0.561
Recall : 0.744
F1 : 0.64

Top 10 model features:


,feature,importance
0,days_with_impressions,0.134951
1,log_impressions_90d,0.129377
2,avg_position,0.109203
3,content_age_days,0.092048
4,char_count,0.038676
5,age_tier_365+,0.036847
6,log_clicks_90d,0.036572
7,word_count,0.035406
8,ctr,0.035156
9,scroll_rate,0.033876



False-positive examples:


,actual,predicted,probability
69,0,1,0.598634
147,0,1,0.572001
168,0,1,0.642586
198,0,1,0.677259
251,0,1,0.524922



False-negative examples:


,actual,predicted,probability
48,1,0,0.381202
279,1,0,0.362311
318,1,0,0.449894
424,1,0,0.384607
477,1,0,0.219944



Interpretation:
The model should be treated as decision-support. Feature importance shows which available signals the Random Forest used most strongly, but it does not establish causation.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.